# Week 5, Session 2: Trees & Traversals (DFS/BFS)

## What You'll Learn
- Understand tree structure: nodes, edges, root, leaves
- Master Binary Tree and BST concepts
- Implement tree traversals: inorder, preorder, postorder
- Understand BFS vs DFS and when to use each

---

## Part 1: Tree Fundamentals

### Tree Terminology

**Tree** = Hierarchical structure with nodes connected by edges.

**Visual Example:**
```
        1        ← Root (depth 0)
       / \
      2   3      ← Children of 1 (depth 1)
     / \
    4   5        ← Leaves (depth 2, no children)
```

**Key Terms:**
- **Root**: Top node (no parent)
- **Node**: Contains data + references to children
- **Leaf**: Node with no children
- **Depth**: Distance from root (root = 0)
- **Height**: Longest path from node to leaf

**Binary Tree:** Each node has at most 2 children (left and right)

In [2]:
# Tree Node using Dictionary
# Each node is a dictionary with 'val', 'left', and 'right' keys
# No classes needed! Just dictionaries!

# Helper function to create a node (optional, but makes code cleaner)
def create_tree_node(val, left=None, right=None):
    """Create a tree node dictionary"""
    return {'val': val, 'left': left, 'right': right}

# Create a simple tree:
#       1
#      / \
#     2   3
#    / \
#   4   5

root = create_tree_node(1)
root['left'] = create_tree_node(2)
root['right'] = create_tree_node(3)
root['left']['left'] = create_tree_node(4)
root['left']['right'] = create_tree_node(5)

# Alternative: Create directly with dictionaries
# root = {'val': 1, 'left': {'val': 2, 'left': {'val': 4, 'left': None, 'right': None}, 
#                            'right': {'val': 5, 'left': None, 'right': None}}, 
#         'right': {'val': 3, 'left': None, 'right': None}}

### Binary Search Tree (BST)

**BST Property:**
- Left subtree contains values < node
- Right subtree contains values > node
- Both subtrees are also BSTs

**What does this mean?** For any node, ALL values in its left subtree are smaller, and ALL values in its right subtree are larger.

**Example - Valid BST:**
```
       4
      / \
     2   6    ← 2 < 4, 6 > 4 ✓
    / \ / \
   1  3 5  7  ← All left values < parent, all right values > parent ✓
```

**Example - Invalid BST:**
```
       4
      / \
     2   6
    / \
   1   5    ← PROBLEM: 5 > 2 (OK), but 5 > 4 (NOT OK!)
```

**Why is this invalid?**
- Node 5 is in the LEFT subtree of node 4
- But 5 > 4, so 5 should be in the RIGHT subtree of node 4!
- Rule: ALL values in left subtree must be < parent
- Since 5 is in left subtree but 5 > 4, this breaks the BST rule ✗

**Advantage:** Enables efficient search (O(log n) average) - we can eliminate half the tree at each step!
- Looking for 5? Start at 4, 5 > 4, so go right (eliminate left half!)
- Looking for 1? Start at 4, 1 < 4, so go left (eliminate right half!)

In [8]:
# BST Example:
#       4
#      / \
#     2   6
#    / \ / \
#   1  3 5  7

bst_root = create_tree_node(4)
bst_root['left'] = create_tree_node(2)
bst_root['right'] = create_tree_node(6)
bst_root['left']['left'] = create_tree_node(1)
bst_root['left']['right'] = create_tree_node(3)
bst_root['right']['left'] = create_tree_node(5)
bst_root['right']['right'] = create_tree_node(7)

# BST Search
def bst_search(root, target):
    if not root or root['val'] == target:
        return root
    
    if target < root['val']:
        return bst_search(root['left'], target)
    else:
        return bst_search(root['right'], target)

# Test
result = bst_search(bst_root, 5)
print(f"Found 5: {result['val'] if result else None}")
print(f"Found 10: {bst_search(bst_root, 10)}")

Found 5: 5
Found 10: None


---

## Part 2: Tree Traversals (DFS)

### Depth-First Search (DFS) Traversals

Three ways to traverse - difference is **when you visit the root**:

**Example Tree:**
```
        1
       / \
      2   3
     / \
    4   5
```

**Preorder: Root → Left → Right**
```
Think: "Visit the node FIRST, then explore its children"

Execution trace (detailed):
  Start at root (1):
    ✓ Visit 1 → Print "1"
    → Go left to node 2:
      ✓ Visit 2 → Print "2"
      → Go left to node 4:
        ✓ Visit 4 → Print "4" (no children, return)
      ← Back to node 2
      → Go right to node 5:
        ✓ Visit 5 → Print "5" (no children, return)
      ← Back to node 2 (done)
    ← Back to root (1)
    → Go right to node 3:
      ✓ Visit 3 → Print "3" (no children, return)
    ← Back to root (1) (done)

Result: 1, 2, 4, 5, 3
Pattern: Always print BEFORE going to children
```

**Inorder: Left → Root → Right**
```
Think: "Explore left FIRST, then visit node, then explore right"

Execution trace (detailed):
  Start at root (1):
    → Go left to node 2 (don't visit 1 yet!):
      → Go left to node 4 (don't visit 2 yet!):
        ✓ Visit 4 → Print "4" (no children, return)
      ← Back to node 2
      ✓ Visit 2 → Print "2" (NOW visit it, between left and right!)
      → Go right to node 5:
        ✓ Visit 5 → Print "5" (no children, return)
      ← Back to node 2 (done)
    ← Back to root (1)
    ✓ Visit 1 → Print "1" (NOW visit it, after left subtree!)
    → Go right to node 3:
      ✓ Visit 3 → Print "3" (no children, return)
    ← Back to root (1) (done)

Result: 4, 2, 5, 1, 3
Pattern: Always print AFTER left subtree, BEFORE right subtree
Note: For BST, inorder gives sorted order!
```

**Postorder: Left → Right → Root**
```
Think: "Explore ALL children FIRST, then visit the node LAST"

Execution trace (detailed):
  Start at root (1):
    → Go left to node 2 (don't visit 1 yet!):
      → Go left to node 4 (don't visit 2 yet!):
        ✓ Visit 4 → Print "4" (no children, return)
      ← Back to node 2
      → Go right to node 5 (don't visit 2 yet!):
        ✓ Visit 5 → Print "5" (no children, return)
      ← Back to node 2
      ✓ Visit 2 → Print "2" (NOW visit it, after BOTH children!)
    ← Back to root (1)
    → Go right to node 3 (don't visit 1 yet!):
      ✓ Visit 3 → Print "3" (no children, return)
    ← Back to root (1)
    ✓ Visit 1 → Print "1" (LAST! After ALL children explored!)

Result: 4, 5, 2, 3, 1
Pattern: Always print AFTER visiting all children
Use case: Good for deleting trees (delete children before parent)
```

**Memory Trick:** Pre=before, In=middle, Post=after (when visiting root)

In [9]:
# Preorder Traversal: Root → Left → Right
# "Pre" = before children, so visit root BEFORE going to children
def preorder(root):
    if not root:  # Base case: empty tree, nothing to do
        return
    
    print(root['val'], end=" ")  # Step 1: Visit root FIRST (before children) - use dictionary key
    preorder(root['left'])       # Step 2: Then visit left subtree - use dictionary key
    preorder(root['right'])      # Step 3: Then visit right subtree - use dictionary key

# Inorder Traversal: Left → Root → Right
# "In" = in the middle, so visit root BETWEEN left and right
def inorder(root):
    if not root:
        return
    
    inorder(root['left'])        # Step 1: Visit left subtree FIRST - use dictionary key
    print(root['val'], end=" ")  # Step 2: Visit root MIDDLE (between left and right) - use dictionary key
    inorder(root['right'])       # Step 3: Then visit right subtree - use dictionary key

# Postorder Traversal: Left → Right → Root
# "Post" = after children, so visit root AFTER visiting children
def postorder(root):
    if not root:
        return
    
    postorder(root['left'])      # Step 1: Visit left subtree FIRST - use dictionary key
    postorder(root['right'])     # Step 2: Visit right subtree SECOND - use dictionary key
    print(root['val'], end=" ")  # Step 3: Visit root LAST (after both children) - use dictionary key

# Test on tree:    1
#                 / \
#                2   3
#               / \
#              4   5

print("Preorder:", end=" ")
preorder(root)  # 1 2 4 5 3
print()

print("Inorder:", end=" ")
inorder(root)  # 4 2 5 1 3
print()

print("Postorder:", end=" ")
postorder(root)  # 4 5 2 3 1
print()

Preorder: 1 2 4 5 3 
Inorder: 4 2 5 1 3 
Postorder: 4 5 2 3 1 


In [10]:
# Iterative Versions (using stacks)

def preorder_iterative(root):
    """Preorder traversal using stack (iterative)"""
    if not root:
        return []
    
    result = []
    stack = [root]
    
    while stack:
        node = stack.pop()
        result.append(node['val'])  # Access 'val' using dictionary key
        
        # Push right first, then left (so left is popped first)
        if node['right']:  # Access 'right' using dictionary key
            stack.append(node['right'])
        if node['left']:  # Access 'left' using dictionary key
            stack.append(node['left'])
    
    return result

def inorder_iterative(root):
    """Inorder traversal using stack (iterative)"""
    result = []
    stack = []
    current = root
    
    while stack or current:
        # Go to leftmost node
        while current:
            stack.append(current)
            current = current['left']  # Access 'left' using dictionary key
        
        # Process node
        current = stack.pop()
        result.append(current['val'])  # Access 'val' using dictionary key
        
        # Move to right
        current = current['right']  # Access 'right' using dictionary key
    
    return result

print(f"Preorder iterative: {preorder_iterative(root)}")
print(f"Inorder iterative: {inorder_iterative(root)}")

Preorder iterative: [1, 2, 4, 5, 3]
Inorder iterative: [4, 2, 5, 1, 3]


---

## Part 3: Breadth-First Search (BFS)

### Level-Order Traversal

**BFS** visits nodes level by level, left to right.

**Uses:**
- Finding shortest path (unweighted graphs)
- Level-order printing
- Finding nodes at specific depth

**Implementation:** Use a queue!

In [ ]:
from collections import deque

def level_order(root):
    """
    BFS: Visit nodes level by level
    """
    if not root:
        return []
    
    result = []
    queue = deque([root])
    
    while queue:
        node = queue.popleft()
        result.append(node.val)
        
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    
    return result

# Test on tree:    1
#                 / \
#                2   3
#               / \
#              4   5

print(f"Level-order: {level_order(root)}")  # [1, 2, 3, 4, 5]

In [ ]:
# Level-order with level separation
def level_order_by_level(root):
    """
    Return list of lists, each list is a level
    """
    if not root:
        return []
    
    result = []
    queue = deque([root])
    
    while queue:
        level_size = len(queue)
        level = []
        
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node['val'])  # Access 'val' using dictionary key
            
            if node['left']:  # Access 'left' using dictionary key
                queue.append(node['left'])
            if node['right']:  # Access 'right' using dictionary key
                queue.append(node['right'])
        
        result.append(level)
    
    return result

print(f"Level-order by level: {level_order_by_level(root)}")
# Output: [[1], [2, 3], [4, 5]]

---

## Part 4: Common Tree Problems

In [ ]:
# Problem 1: Maximum Depth (Height)
def max_depth(root):
    """Find maximum depth of tree"""
    if not root:
        return 0
    
    left_depth = max_depth(root['left'])  # Access 'left' using dictionary key
    right_depth = max_depth(root['right'])  # Access 'right' using dictionary key
    
    return 1 + max(left_depth, right_depth)

print(f"Max depth: {max_depth(root)}")

# Problem 2: Check if Same Tree
def is_same_tree(p, q):
    """Check if two trees are identical"""
    if not p and not q:
        return True
    if not p or not q:
        return False
    
    return (p['val'] == q['val'] and  # Access 'val' using dictionary key
            is_same_tree(p['left'], q['left']) and  # Access 'left' using dictionary key
            is_same_tree(p['right'], q['right']))  # Access 'right' using dictionary key

# Problem 3: Invert Binary Tree
def invert_tree(root):
    """Invert (mirror) a binary tree"""
    if not root:
        return None
    
    # Swap children
    root['left'], root['right'] = root['right'], root['left']  # Access using dictionary keys
    
    # Recursively invert subtrees
    invert_tree(root['left'])
    invert_tree(root['right'])
    
    return root

---

## Part 5: Practice Problems

### Problem 1: Validate Binary Search Tree
Check if a binary tree is a valid BST.

**Hint:** Use inorder traversal or check bounds

In [ ]:
# Your solution here:
def is_valid_bst(root):
    # TODO: Implement
    pass

### Problem 2: Symmetric Tree
Check if a tree is symmetric (mirror of itself).

**Example:**
```
    1
   / \
  2   2
 / \ / \
3  4 4  3
```
This is symmetric!

In [ ]:
# Your solution here:
def is_symmetric(root):
    # TODO: Implement
    pass

### Problem 3: Path Sum
Check if there's a root-to-leaf path with given sum.

**Example:** Tree `[5,4,8,11,null,13,4,7,2,null,null,null,1]`, sum=22 → True
(Path: 5→4→11→2)

In [ ]:
# Your solution here:
def has_path_sum(root, target_sum):
    # TODO: Implement
    pass

---

## Solutions (Try first!)

<details>
<summary>Click to reveal solutions</summary>

### Solution 1: Validate BST
```python
def is_valid_bst(root):
    """Validate BST using dictionary nodes"""
    def validate(node, min_val, max_val):
        if not node:
            return True
        
        if node['val'] <= min_val or node['val'] >= max_val:  # Access 'val' using dictionary key
            return False
        
        return (validate(node['left'], min_val, node['val']) and  # Access 'left' using dictionary key
                validate(node['right'], node['val'], max_val))  # Access 'right' using dictionary key
    
    return validate(root, float('-inf'), float('inf'))
```

### Solution 2: Symmetric Tree
```python
def is_symmetric(root):
    """Check if tree is symmetric using dictionary nodes"""
    def is_mirror(left, right):
        if not left and not right:
            return True
        if not left or not right:
            return False
        
        return (left['val'] == right['val'] and  # Access 'val' using dictionary key
                is_mirror(left['left'], right['right']) and  # Access 'left'/'right' using dictionary keys
                is_mirror(left['right'], right['left']))
    
    if not root:
        return True
    return is_mirror(root['left'], root['right'])  # Access 'left'/'right' using dictionary keys
```

### Solution 3: Path Sum
```python
def has_path_sum(root, target_sum):
    """Check if path sum exists using dictionary nodes"""
    if not root:
        return False
    
    if not root['left'] and not root['right']:  # Access 'left'/'right' using dictionary keys
        return root['val'] == target_sum  # Access 'val' using dictionary key
    
    remaining = target_sum - root['val']  # Access 'val' using dictionary key
    return (has_path_sum(root['left'], remaining) or  # Access 'left' using dictionary key
            has_path_sum(root['right'], remaining))  # Access 'right' using dictionary key
```

</details>

---

## Key Takeaways

✅ **Tree Structure:**
- Hierarchical with root, nodes, edges
- Binary tree: max 2 children per node
- BST: left < node < right

✅ **DFS Traversals:**
- Preorder: Root → Left → Right
- Inorder: Left → Root → Right (gives sorted order for BST)
- Postorder: Left → Right → Root
- Can be recursive or iterative (with stack)

✅ **BFS (Level-Order):**
- Visit level by level
- Use queue
- Good for shortest path problems

✅ **When to Use:**
- DFS: When you need to explore deep (path problems, tree properties)
- BFS: When you need level-by-level (shortest path, level problems)

---

## Homework

1. Complete all practice problems
2. Solve LeetCode Easy:
   - [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/)
   - [226. Invert Binary Tree](https://leetcode.com/problems/invert-binary-tree/)
   - [101. Symmetric Tree](https://leetcode.com/problems/symmetric-tree/)
3. Practice drawing trees and tracing traversals manually

****Next Session:** Week 6, Session 1 - Heaps & Graphs Basics